# 06 Forecasting — Capacity Buildout

Two forecasting approaches:
1. **XGBoost / RF cross-sectional model** — predicts 2035 capacity per country using fleet and economic features.
2. **Statsmodels Exponential Smoothing** — global capacity trend extrapolation from historical commissioning data.

> Run the pipeline first: `python run_pipeline.py`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
processed = ROOT / 'data' / 'processed'
predictions = ROOT / 'outputs' / 'predictions'

import json
countries = pd.read_csv(processed / 'country_nuclear_profile.csv')
forecasts = pd.read_csv(predictions / 'capacity_forecasts.csv')
global_fc = pd.read_csv(predictions / 'statsmodels_global_capacity_forecast.csv')
metrics = json.loads((ROOT / 'outputs' / 'metrics' / 'forecasting_metrics.json').read_text())
print(json.dumps(metrics, indent=2))

## Model context

**Target variable:** `target_capacity_2035_mwe` — a scenario-derived heuristic: `operating + 0.75×construction + 0.45×planned + 0.15×proposed`.

This is not an independent ground truth — it's a formula of the same features the model uses. The model serves as a **pipeline and feature-importance demonstration**, not a production forecast. For real use, you'd need historical capacity snapshots at different points in time to build a proper supervised dataset.

## Forecast vs heuristic target by country

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
x = range(len(forecasts))
ax.bar(x, forecasts['target_capacity_2035_mwe']/1000, alpha=0.6, label='Heuristic target (GWe)', color='steelblue')
ax.bar(x, forecasts['forecast_capacity_2035_mwe']/1000, alpha=0.6, label='Model forecast (GWe)', color='orange')
ax.set_xticks(list(x)); ax.set_xticklabels(forecasts['country'], rotation=40, ha='right')
ax.set_title('2035 Capacity: Heuristic Target vs Model Forecast', fontsize=13)
ax.set_ylabel('GWe'); ax.legend()
plt.tight_layout(); plt.show()

## Feature importance

In [ ]:
from joblib import load
import numpy as np

model_path = ROOT / 'models' / 'capacity_forecast_xgboost.joblib'
pipe = load(model_path)

pre = pipe.named_steps['preprocess']
model = pipe.named_steps['model']

cat_features = list(pre.named_transformers_['cat'].get_feature_names_out(['region','income_group']))
num_features = ['gdp_current_usd','population','electricity_generation_twh','nuclear_share_percent',
                'operating_capacity_mwe','construction_capacity_mwe','planned_capacity_mwe',
                'proposed_capacity_mwe','average_fleet_age','nuclear_experience_years','policy_signal_score']
all_features = cat_features + num_features

try:
    importances = model.feature_importances_
    imp = pd.DataFrame({'feature': all_features[:len(importances)], 'importance': importances})
    imp = imp.sort_values('importance', ascending=False).head(15)
    
    fig, ax = plt.subplots(figsize=(10,5))
    sns.barplot(data=imp, x='importance', y='feature', ax=ax, palette='Blues_r')
    ax.set_title('Top 15 Feature Importances', fontsize=13)
    plt.tight_layout(); plt.show()
except AttributeError:
    print("Feature importances not available for this model type.")

## Global capacity trend — Exponential Smoothing

In [ ]:
reactors_all = pd.read_csv(processed / 'reactors_master.csv', parse_dates=['commercial_operation_date'])
historical = (
    reactors_all[reactors_all.commercial_operation_date.notna()]
    .assign(year=lambda df: df.commercial_operation_date.dt.year)
    .groupby('year')['capacity_mwe']
    .sum()
    .sort_index()
    .cumsum()
    .div(1000)
)

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(historical.index, historical.values, 'o-', label='Historical cumulative GWe', color='steelblue')
ax.plot(global_fc['year'], global_fc['global_capacity_mwe_forecast']/1000, 's--', label='Forecast (Holt-Winters)', color='orange')
ax.set_title('Global Cumulative Nuclear Capacity – Historical + Forecast', fontsize=13)
ax.set_ylabel('Cumulative GWe'); ax.set_xlabel('Year')
ax.legend(); plt.tight_layout(); plt.show()

**Limitations of the Exponential Smoothing forecast:**
- Sample data has sparse historical coverage (few years with commissionings).
- The model extrapolates the recent trend, which is sensitive to the last few data points.
- A production model would use 70+ years of PRIS data, policy-informed scenarios, and country-level breakdowns.

## What a production forecasting pipeline would look like

In [ ]:
# This cell documents the ideal approach — not executable without historical snapshot data.
"""
Production setup:
1. Pull IAEA PRIS annual snapshots (2000–2025) — capacity per country per year.
2. Build features at the country-year level (GDP, policy changes, fleet age, construction pipeline).
3. Use a panel data model (e.g. LightGBM with country fixed effects, or ARIMA per country).
4. Validate with time-series cross-validation: train on 2000–2015, test on 2016–2020.
5. Evaluate with MAE on held-out years, not random splits.

Current approach is a cross-sectional demo — useful for showing the full sklearn pipeline
pattern (ColumnTransformer, Pipeline, joblib) but not a real forecasting model.
"""
print("See docstring above for production design.")